# DWM Lab 6 — Naive Bayes Classification

### Dataset: Online Retail

**Objective:**  
To implement a **Naive Bayes Classifier** on the Online Retail dataset and evaluate the classification model using accuracy, confusion matrix, precision, recall and F1-score.

### Dataset adaptation

The original lab applies Gaussian Naive Bayes to retail/e-commerce data and classifies transactions into sales categories.

For this common Online Retail dataset:

- `Quantity`
- `UnitPrice`
- `Month`
- `SalesAmount`

are used to describe a transaction.

A target variable called **Sales_Category** is created from `SalesAmount`:

- **Low**
- **Medium**
- **High**

`SalesAmount` is used to create the target and is therefore **not used directly as an input feature**, avoiding direct target leakage.

## 1. Import Required Libraries

Pandas is used for data handling, Matplotlib/Seaborn for visualization, and Scikit-learn for Gaussian Naive Bayes and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

sns.set_theme(style="whitegrid")

## 2. Load the Dataset

Load the same `Online Retail.xlsx` dataset used in the previous DWM experiments.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Preprocessing

The following preprocessing steps are performed:

1. Convert `InvoiceDate` into datetime format.
2. Remove cancelled invoices.
3. Remove transactions with non-positive quantity or unit price.
4. Calculate `SalesAmount`.
5. Extract the month from `InvoiceDate`.
6. Create the `Sales_Category` target.
7. Select the numerical features required by Gaussian Naive Bayes.

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid sales transactions
df = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()

# Calculate transaction sales
df['SalesAmount'] = df['Quantity'] * df['UnitPrice']

# Extract month
df['Month'] = df['InvoiceDate'].dt.month

# Create Low / Medium / High target categories using tertiles
q1 = df['SalesAmount'].quantile(1/3)
q2 = df['SalesAmount'].quantile(2/3)

df['Sales_Category'] = pd.cut(
    df['SalesAmount'],
    bins=[-np.inf, q1, q2, np.inf],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

# Prepare modelling dataset
model_df = df[
    ['Quantity', 'UnitPrice', 'Month', 'Sales_Category']
].dropna()

print("Cleaned dataset shape:", model_df.shape)
print("\nClass distribution:")
print(model_df['Sales_Category'].value_counts())

display(model_df.head())

## 4. Prepare Features and Target

Features:

- `Quantity`
- `UnitPrice`
- `Month`

Target:

- `Sales_Category`

The data is divided into **70% training** and **30% testing** data, matching the train-test approach used in the lab.

In [ ]:
X = model_df[['Quantity', 'UnitPrice', 'Month']]
y = model_df['Sales_Category']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 5. Gaussian Naive Bayes

### Theory

Naive Bayes is a probabilistic classification algorithm based on Bayes' theorem.

**Bayes' theorem:**

P(C|X) = P(X|C) P(C) / P(X)

For Gaussian Naive Bayes, continuous features are assumed to follow a Gaussian (normal) distribution within each class.

The model assumes that the features are conditionally independent given the class.

In [ ]:
# Create Gaussian Naive Bayes classifier
nb_model = GaussianNB()

# Train the model
nb_model.fit(X_train, y_train)

# Predict test data
y_pred = nb_model.predict(X_test)

print("Gaussian Naive Bayes model trained successfully.")

## 6. Model Accuracy

Accuracy is the proportion of correctly classified observations among all observations.

**Accuracy = Correct Predictions / Total Predictions**

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

## 7. Confusion Matrix

A confusion matrix shows the number of correct and incorrect predictions for each class.

Rows represent the actual classes and columns represent the predicted classes.

In [ ]:
labels = ['Low', 'Medium', 'High']

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=labels,
    yticklabels=labels
)
plt.title('Naive Bayes Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.tight_layout()
plt.show()

## 8. Precision, Recall and F1-Score

### Precision
Precision measures how many observations predicted as a class actually belong to that class.

**Precision = TP / (TP + FP)**

### Recall
Recall measures how many actual observations of a class were correctly identified.

**Recall = TP / (TP + FN)**

### F1-Score
F1-score is the harmonic mean of precision and recall.

**F1 = 2 × Precision × Recall / (Precision + Recall)**

In [ ]:
precision = precision_score(
    y_test, y_pred,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    y_test, y_pred,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    y_test, y_pred,
    average='weighted',
    zero_division=0
)

print("Weighted Precision:", precision)
print("Weighted Recall:", recall)
print("Weighted F1-Score:", f1)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 9. Compare Actual and Predicted Classes

Display a sample of actual and predicted sales categories.

In [ ]:
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

display(results.head(20))

## 10. Predict a New Transaction

Use the trained Gaussian Naive Bayes model to classify a new transaction.

Example:
- Quantity = 10
- Unit Price = 5
- Month = 12

In [ ]:
new_transaction = pd.DataFrame({
    'Quantity': [10],
    'UnitPrice': [5],
    'Month': [12]
})

prediction = nb_model.predict(new_transaction)
probabilities = nb_model.predict_proba(new_transaction)

print("New Transaction:")
display(new_transaction)

print("Predicted Sales Category:", prediction[0])

probability_df = pd.DataFrame(
    probabilities,
    columns=nb_model.classes_
)

print("\nClass probabilities:")
display(probability_df)

## 11. Summary of Evaluation Metrics

Create a compact table containing the main performance measures.

In [ ]:
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1-Score'],
    'Score': [accuracy, precision, recall, f1]
})

display(metrics_df)

## Conclusion

Gaussian Naive Bayes was successfully implemented on the Online Retail dataset.

The experiment included:

- Data preprocessing
- Creation of Low, Medium and High sales categories
- 70:30 train-test split
- Gaussian Naive Bayes classification
- Accuracy calculation
- Confusion matrix
- Precision
- Recall
- F1-score
- Classification report
- Prediction of a new transaction

The same Online Retail dataset can be reused for the next DWM experiments involving clustering and association rule mining.